# Sanity Check - Step 00: Downsample

Überprüft:
- Downsampling erfolgreich durchgeführt
- Sampling Rate korrekt reduziert
- Datenlänge und -größe stimmen
- Keine Artefakte durch Downsampling

In [ ]:
import os
import sys
from pathlib import Path
import mne
import numpy as np
from mne_bids import BIDSPath, read_raw_bids

root = Path.cwd()
candidate_roots = [root, root.parent, root.parent.parent]
for candidate in candidate_roots:
    pipeline_dir = candidate / "eeg_pipeline"
    if pipeline_dir.exists() and str(pipeline_dir) not in sys.path:
        sys.path.insert(0, str(pipeline_dir))
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

try:
    from eeg_pipeline import config
except ModuleNotFoundError:
    import config

print("Setup erfolgreich")

Setup erfolgreich


## 1. Originaldaten (BIDS)

In [ ]:
subject_id = os.getenv("EEG_SUBJECT", config.SUBJECTS[0]).strip()

bids_path = BIDSPath(
    subject=subject_id,
    task="RPS",
    datatype='eeg',
    suffix='eeg',
    root=config.BIDS_ROOT
)

raw_original = read_raw_bids(bids_path, verbose=False)

print(f"Nutze subject: sub-{subject_id}")
print(f"\n=== ORIGINALDATEN ===")
print(f"Anzahl Kanäle: {len(raw_original.ch_names)}")
print(f"Sampling Rate: {raw_original.info['sfreq']} Hz")
print(f"Dauer: {raw_original.times[-1]:.2f} Sekunden")

n_samples = raw_original.n_times
n_channels = len(raw_original.ch_names)
estimated_size_mb = (n_channels * n_samples * 8) / 1e6
print(f"Geschätzte Datengröße: {estimated_size_mb:.2f} MB")


=== ORIGINALDATEN ===
Anzahl Kanäle: 143
Sampling Rate: 2048.0 Hz
Dauer: 3669.00 Sekunden
Geschätzte Datengröße: 8596.14 MB


C:\Users\BKALYON\AppData\Local\Temp\ipykernel_28440\1692125182.py:11: RuntimeWarning: Did not find any channels.tsv associated with sub-01_task-RPS.

The search_str was "c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\MNE-sample-data\ds006761\sub-01\**\eeg\sub-01*channels.tsv"
  raw_original = read_raw_bids(bids_path, verbose=False)
C:\Users\BKALYON\AppData\Local\Temp\ipykernel_28440\1692125182.py:11: RuntimeWarning: participants.tsv file not found for c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\MNE-sample-data\ds006761\sub-01\eeg\sub-01_task-RPS_eeg.bdf
  raw_original = read_raw_bids(bids_path, verbose=False)


## 2. Downsampled Daten (Step 00 Output)

In [7]:
out_path = config.OUTPUT_DIR / f"sub-{subject_id}_downsampled.fif"
if not out_path.exists():
    print(f"Fehler: Downsampled file nicht gefunden: {out_path}")
else:
    raw_downsampled = mne.io.read_raw_fif(str(out_path), preload=False)
    
    print(f"\n=== NACH DOWNSAMPLING ===")
    print(f"Sampling Rate: {raw_downsampled.info['sfreq']} Hz")
    print(f"Dauer: {raw_downsampled.times[-1]:.2f} Sekunden")
    print(f"Anzahl Kanäle: {len(raw_downsampled.ch_names)}")
    
    n_samples_ds = raw_downsampled.n_times
    estimated_size_mb_ds = (n_channels * n_samples_ds * 8) / 1e6
    print(f"Geschätzte Datengröße: {estimated_size_mb_ds:.2f} MB")
    
    print(f"\n=== VERGLEICH ===")
    print(f"Downsampling Faktor: {raw_original.info['sfreq'] / raw_downsampled.info['sfreq']:.1f}x")
    print(f"Größenreduktion: {estimated_size_mb_ds / estimated_size_mb * 100:.1f}% der Originalgröße")

Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_downsampled.fif...
Isotrak not found


C:\Users\BKALYON\AppData\Local\Temp\ipykernel_28440\2786471368.py:5: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_downsampled.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_downsampled = mne.io.read_raw_fif(str(out_path), preload=False)


    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.

=== NACH DOWNSAMPLING ===
Sampling Rate: 200.0 Hz
Dauer: 3668.99 Sekunden
Anzahl Kanäle: 143
Geschätzte Datengröße: 839.47 MB

=== VERGLEICH ===
Downsampling Faktor: 10.2x
Größenreduktion: 9.8% der Originalgröße


## 3. Validierungen

In [8]:
print("=== SANITY CHECKS ===")

# Kanal-Anzahl
if len(raw_original.ch_names) == len(raw_downsampled.ch_names):
    print("✓ Kanal-Anzahl erhalten")
else:
    print(f"✗ Kanal-Anzahl geändert: {len(raw_original.ch_names)} -> {len(raw_downsampled.ch_names)}")

# Dauer prüfen
original_duration = raw_original.times[-1]
downsampled_duration = raw_downsampled.times[-1]
duration_diff = abs(original_duration - downsampled_duration)

if duration_diff < 0.1:
    print(f"✓ Dauer erhalten (Differenz: {duration_diff:.4f}s)")
else:
    print(f"✗ Dauer geändert: {original_duration:.2f}s -> {downsampled_duration:.2f}s")

# Sampling Rate
if raw_downsampled.info['sfreq'] == config.DOWNSAMPLE_SFREQ:
    print(f"✓ Sampling Rate korrekt: {config.DOWNSAMPLE_SFREQ} Hz")
else:
    print(f"⚠ Sampling Rate: Expected {config.DOWNSAMPLE_SFREQ} Hz, got {raw_downsampled.info['sfreq']} Hz")

=== SANITY CHECKS ===
✓ Kanal-Anzahl erhalten
✓ Dauer erhalten (Differenz: 0.0045s)
✓ Sampling Rate korrekt: 200 Hz
